In [249]:
from urllib.parse import urlparse
import seaborn as sns
import matplotlib.ticker as ticker
import pandas as pd
import matplotlib.pyplot as plt
import json
import re
from textwrap import wrap
import numpy as np

In [250]:
def convert_list_columns_to_sets(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert all list-valued columns in a DataFrame to sets
    (removes duplicates in each cell).
    """
    df_copy = df.copy()
    for col in df_copy.columns:
        if df_copy[col].apply(lambda x: isinstance(x, list)).any():
            df_copy[col] = df_copy[col].apply(lambda x: list(set(x)) if isinstance(x, list) else x)
    return df_copy

In [251]:
def unify_imaging_methods(lst):
    if not isinstance(lst, list):
        return [lst]
    res = []
    for l in lst:
        if l in ["Camera (manually triggered; e.g. Smartphone, System Camera, SLR Camera)",
                 "Video Camera (e.g. CCTV Camera, Action Camera, PTZ Camera)"]:
            # res.append("Camera")
            res.append("Non-specialized Camera")
        elif l in [
                 "Time-lapse Camera",
                 "Event Camera"]:
            res.append("Other")
        elif l in ["Camera Trap (temperature- or motion triggered)", "Camera Trap"]:
            res.append("Camera Trap")
        else:
            res.append(l)
    if len(res) == 0:
        res.append("Non-specialized Camera")
        # res.append("Unknown")
    return list(set(res))

In [252]:
IUCN_MAPPING = {
    "Forest": "Forest",
    "Savanna": "Savanna",
    "Shrubland": "Shrubland",
    "Grassland": "Grassland",
    "WetlandsInland": "Wetlands (inland)",
    "Desert": "Desert",
    "RockyAreasEGInlandCliffsMountainPeaks": "Rocky areas (e.g. inland cliffs, mountain peaks)",
    "CavesSubterraneanHabitatsNonAquatic": "Cave and subterranean habitats (non-aquatic)",

    # Marine
    # "MarineNeritic": "Marine Neritic",
    # "MarineOceanic": "Marine Oceanic",
    # "MarineCoastalSupratidal": "Marine Coastal/Supratidal",
    # "MarineDeepOceanFloorBenthicAndDemersal": "Marine Deep Ocean Floor (Benthic and Demersal)",
    # "MarineIntertidal": "Marine Intertidal",
    "MarineNeritic": "Marine",
    "MarineOceanic": "Marine",
    "MarineCoastalSupratidal": "Marine",
    "MarineDeepOceanFloorBenthicAndDemersal": "Marine",
    "MarineIntertidal": "Marine",

    # Artificial
    "ArtificialTerrestrial": "Artificial Terrestrial",
    "ArtificialAquatic": "Artificial Aquatic",

    # Unknown
    "Unknown": "Unknown / Other"
}

def map_to_first_level(habitat: str) -> str:
    """Maps a list of IUCN second/third level habitats to first-level habitats"""
    return IUCN_MAPPING.get(habitat, "Unknown")

def map_all_to_first_level(habitats: list[str]) -> list[str]:
    res = []
    for habitat in habitats:
        res.append(map_to_first_level(habitat))
    return list(set(res))

In [253]:
def plot_side_by_side(
        df1, df2,
        index_col1,
        index_col2,
        df1_name="Manuel",
        df2_name="Auto",
        relative: bool=True,
        figsize=(12, 6),
        title="",
        top_n: int=None,
        font_size: int = 15,
        target_path: str = None,
        label_wrap_clip: int = 20,
        bar_width=0.4
):
    # --- compute counts (absolute + relative) ---
    auto_abs = df1[index_col1].value_counts(dropna=False)
    ma_abs   = df2[index_col2].value_counts(dropna=False)

    auto_rel = auto_abs / auto_abs.sum()
    ma_rel   = ma_abs / ma_abs.sum()

    # pick what to plot
    series_left  = ma_rel if relative else ma_abs
    series_right = auto_rel if relative else auto_abs

    # combine into df
    df_plot = pd.DataFrame({
        df1_name: series_left,
        df2_name: series_right,
    }).fillna(0)

    abs_for_labels = pd.DataFrame({
        df1_name: ma_abs.reindex(df_plot.index).fillna(0).astype(int),
        df2_name: auto_abs.reindex(df_plot.index).fillna(0).astype(int),
    })

    # # --- apply top_n filtering ---
    # if top_n is not None:
    #     # rank by sum of both series, then take top_n
    #     order = df_plot.sum(axis=1).sort_values(ascending=False).head(top_n).index
    #     df_plot = df_plot.loc[order]
    #     abs_for_labels = abs_for_labels.loc[order]
    #
    # else:
    #     df_plot = df_plot.sort_index()
    #     abs_for_labels = abs_for_labels.loc[df_plot.index]
    totals = df_plot.sum(axis=1)
    order_all = totals.sort_values(ascending=False).index
    order = order_all[:top_n] if top_n is not None else order_all

    df_plot = df_plot.loc[order]
    abs_for_labels = abs_for_labels.loc[order]

    # --- plotting ---
    fig, ax = plt.subplots(figsize=figsize)
    colors = [plt.cm.tab20.colors[0], plt.cm.tab20.colors[12]]
    df_plot.plot(kind="bar", width=bar_width, ax=ax, color=colors)

    # y-axis formatting
    if relative:
        ax.yaxis.set_major_locator(ticker.MultipleLocator(0.1))
        ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0))
        ax.set_ylabel("Relative share (%)", fontsize=font_size)

        # find maximum relative height across both data series
        ymax = df_plot.max().max()
        ax.set_ylim(0, ymax * 1.1)   # 10% headroom above data
    else:
        ax.set_ylabel("Count", fontsize=font_size)
        ymax = df_plot.max().max()
        ax.set_ylim(0, ymax * 1.1)

    ax.set_title(title)
    import textwrap
    wrapped_labels = [textwrap.fill(str(l), label_wrap_clip) for l in df_plot.index]
    ax.set_xticklabels(wrapped_labels, rotation=90, fontsize=font_size)

    # --- absolute counts above bars when relative ---
    if relative:
        ymax = df_plot.max().max()
        margin = (ymax * 0.05) if not relative else max(0.05, ymax * 0.05)
        ax.set_ylim(0, ymax + margin)

        for col_idx, container in enumerate(ax.containers):
            for i, rect in enumerate(container):
                height = rect.get_height()
                count_str = str(abs_for_labels.iloc[i, col_idx])
                if height > 0:
                    ax.text(
                        rect.get_x() + rect.get_width() / 2, height + (ymax * 0.01),
                        count_str,
                        ha="center", va="bottom",
                        rotation=90,
                        clip_on=False
                    )
    plt.tick_params(axis="y", labelsize=font_size)
    plt.xlabel("")
    ax.margins(x=0.01)
    ax.legend(title="")
    plt.tight_layout()
    if target_path is not None:
        plt.savefig(target_path)
    plt.show()


In [254]:
def plot_side_by_side_stacked(
    df1, df2,
    index_col1, column_col1,
    index_col2, column_col2,
    relative: bool = True,
    figsize=(30, 12),
    palette_name="tab20",
    title="",
    legend_settings: dict = None,
    font_size: int = 15,
    target_path: str = None,
    bar_width=0.4,
    add_absolute: bool = True,
):
    """
    Plot two stacked bar charts side by side for two DataFrames.

    Parameters
    ----------
    df1, df2 : pandas.DataFrame
        Input DataFrames (long format).
    index_col1, column_col1 : str
        Column names in df1 for index and pivot columns.
    index_col2, column_col2 : str
        Column names in df2 for index and pivot columns.
    relative : bool, default=True
        If True, plot relative shares with absolute counts shown below bars.
        If False, plot absolute values.
    figsize : tuple
        Figure size for matplotlib.
    palette_name : str
        Seaborn palette name.
    """

    # --- Pivot absolute counts ---
    pivot1_abs = (
        df1.pivot_table(index=index_col1, columns=column_col1,
                        aggfunc="size", fill_value=0)
    )
    pivot2_abs = (
        df2.pivot_table(index=index_col2, columns=column_col2,
                        aggfunc="size", fill_value=0)
    )

    if relative:
        pivot1 = pivot1_abs.div(pivot1_abs.sum(axis=1), axis=0).fillna(0)
        pivot2 = pivot2_abs.div(pivot2_abs.sum(axis=1), axis=0).fillna(0)
    else:
        pivot1, pivot2 = pivot1_abs.copy(), pivot2_abs.copy()

    # --- Align indices and columns ---
    all_rows = sorted(set(pivot1.index) | set(pivot2.index))
    all_cols = sorted(set(pivot1.columns) | set(pivot2.columns))

    pivot1 = pivot1.reindex(index=all_rows, columns=all_cols, fill_value=0)
    pivot2 = pivot2.reindex(index=all_rows, columns=all_cols, fill_value=0)
    pivot1_abs = pivot1_abs.reindex(index=all_rows, columns=all_cols, fill_value=0)
    pivot2_abs = pivot2_abs.reindex(index=all_rows, columns=all_cols, fill_value=0)

    # --- Plot ---
    fig, ax = plt.subplots(figsize=figsize)
    x = np.arange(len(all_rows))

    bottom1 = np.zeros(len(all_rows))
    bottom2 = np.zeros(len(all_rows))

    palette = sns.color_palette(palette_name, n_colors=len(all_cols))
    colors = {col: palette[i] for i, col in enumerate(all_cols)}

    for col in all_cols:
        ax.bar(x - bar_width/2, pivot1[col].values, bar_width,
               bottom=bottom1, color=colors[col], label=col)
        bottom1 += pivot1[col].values

        ax.bar(x + bar_width/2, pivot2[col].values, bar_width,
               bottom=bottom2, color=colors[col])
        bottom2 += pivot2[col].values

    # --- Absolute numbers below bars (only if relative) ---
    if relative and add_absolute:
        ax.yaxis.set_major_formatter(ticker.PercentFormatter(1.0))
        for i, row in enumerate(all_rows):
            total1 = pivot1_abs.loc[row].sum()
            total2 = pivot2_abs.loc[row].sum()
            ax.text(x[i] - bar_width/2, -0.05, str(total1),
                    ha="center", va="top", fontsize=font_size, rotation=90)
            ax.text(x[i] + bar_width/2, -0.05, str(total2),
                    ha="center", va="top", fontsize=font_size, rotation=90)

    # Cosmetics
    ax.set_xticks(x)
    import textwrap
    wrapped_labels = [textwrap.fill(l, 10) for l in all_rows]
    ax.set_xticklabels(wrapped_labels, rotation=90, fontsize=font_size)
    ax.set_ylabel("Relative share (%) per bar" if relative else "Absolute counts", fontsize=font_size)
    plt.yticks(fontsize=font_size)
    ax.set_title(title)
    if legend_settings is None:
        ax.legend(title=column_col1, bbox_to_anchor=(1.02, 1), loc="upper left", font_size=font_size)
    else:
        ax.legend(**legend_settings)
    if relative:
        ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0))
    ax.margins(x=0.01)
    ax.set_ylim(-0.19 if relative and add_absolute else 0, 1.05 if relative else None)
    plt.tight_layout()
    if target_path is not None:
        plt.savefig(target_path)
    plt.show()

In [255]:
from collections import Counter
import pandas as pd
import re

def _norm_name(s: str) -> str:
    """Normalize a species/taxon string for matching: lowercase, collapse spaces."""
    return re.sub(r"\s+", " ", s.strip().lower())

def build_lookup_from_hits(hits: dict):
    """
    Build a lookup: normalized name -> match dict (and keep usageKey to deduplicate).
    We index by:
      - the original key in 'hits'
      - match.canonicalName
      - match.scientificName
      - originalQuery (if present)
    """
    name_to_match = {}
    usagekey_to_match = {}
    for key_name, payload in hits.items():
        m = payload.get("match", {}) or {}
        usage_key = m.get("usageKey")
        if usage_key is not None and usage_key not in usagekey_to_match:
            usagekey_to_match[usage_key] = m

        candidates = {
            key_name,
            m.get("canonicalName", ""),
            m.get("scientificName", ""),
            payload.get("originalQuery", ""),
        }
        for c in candidates:
            if not c:
                continue
            name_to_match[_norm_name(c)] = m

    return name_to_match, usagekey_to_match

def taxonomy_counters_from_rows(
    matched_names_per_row,
    name_to_match,
    restrict_kingdom=None,
    unique_per_row=True,
):
    """
    Compute per-rank counters.

    If unique_per_row=True:
      For each row, collect UNIQUE values at each rank (kingdom..species),
      then increment each of those values ONCE for that row.

    If unique_per_row=False (original behavior):
      Count every matched name (species) toward its ranks.
    """
    rank_fields = ["kingdom", "phylum", "class", "order", "family", "genus", "species"]
    per_rank = {r: Counter() for r in rank_fields}

    if unique_per_row:
        # Row-wise unique counting
        for row_names in matched_names_per_row:
            row_unique_by_rank = {r: set() for r in rank_fields}
            for raw_name in row_names:
                nm = _norm_name(raw_name)
                m = name_to_match.get(nm, {})
                if not m:
                    continue
                if restrict_kingdom and m.get("kingdom") != restrict_kingdom:
                    continue
                for r in rank_fields:
                    val = m.get(r)
                    if val:
                        row_unique_by_rank[r].add(val)
            # increment once per unique rank value in this row
            for r in rank_fields:
                for val in row_unique_by_rank[r]:
                    per_rank[r][val] += 1
    else:
        # Original flat counting (every species contributes)
        for row_names in matched_names_per_row:
            for raw_name in row_names:
                nm = _norm_name(raw_name)
                m = name_to_match.get(nm, {})
                if not m:
                    continue
                if restrict_kingdom and m.get("kingdom") != restrict_kingdom:
                    continue
                for r in rank_fields:
                    val = m.get(r)
                    if val:
                        per_rank[r][val] += 1

    per_rank_occurrence_totals = {r: sum(c.values()) for r, c in per_rank.items()}
    per_rank_unique_taxa = {r: len(c) for r, c in per_rank.items()}

    tidy = pd.DataFrame({
        "rank": per_rank.keys(),
        "occurrences": [per_rank_occurrence_totals[r] for r in per_rank.keys()],
        "unique_taxa": [per_rank_unique_taxa[r] for r in per_rank.keys()],
    }).set_index("rank")

    return per_rank, per_rank_occurrence_totals, per_rank_unique_taxa, tidy

def compute_taxonomy_counts(
    df: pd.DataFrame,
    hits_json: dict,
    columns=("Species (Text)(translated) - verified", "Species (Images)(translated) - verified"),
    restrict_kingdom="Animalia",
    unique_per_row=True,   # NEW: True => count each rank only once per row
    return_details=False
):
    hits = hits_json.get("hits", {}) or {}
    name_to_match, usagekey_to_match = build_lookup_from_hits(hits)

    matched_names_per_row = []
    unmatched = []

    for _, row in df.iterrows():
        # unique raw species names within this row across the specified columns
        species_names = set()
        for col in columns:
            # assumes each cell is an iterable of names; adjust if it's strings/NaN, etc.
            species_names.update(row[col])

        row_matched = []
        for raw_name in species_names:
            nm = _norm_name(raw_name)
            m = name_to_match.get(nm)
            if m:
                row_matched.append(raw_name)  # keep original raw name
            else:
                unmatched.append(raw_name)

        matched_names_per_row.append(row_matched)

    per_rank, totals, unique_counts, tidy = taxonomy_counters_from_rows(
        matched_names_per_row,
        name_to_match,
        restrict_kingdom=restrict_kingdom,
        unique_per_row=unique_per_row,
    )

    if return_details:
        # flatten matched names for convenience in diagnostics
        matched_flat = [n for row_names in matched_names_per_row for n in row_names]
        return {
            "per_rank_counters": per_rank,
            "totals_by_rank": totals,
            "unique_taxa_by_rank": unique_counts,
            "tidy_counts_df": tidy,
            "matched_names": matched_flat,
            "unmatched_names": unmatched,
        }
    else:
        return tidy


In [256]:
def plot_rank_counts_compare(
    per_rank_counters_a,
    per_rank_counters_b,
    rank="class",
    label_a="DF1",
    label_b="DF2",
    top_n=None,
    min_count=None,      # filter on RAW counts (pre-normalization)
    min_prop=None,       # filter on PROPORTIONS (post-normalization)
    sort_by="sum",       # "sum" | "a" | "b" (applied to whatever is on the y-axis: counts or proportions)
    normalize=False,     # if True, bars show proportions
    title=None,
    figsize=(10, 4),
    annotate=True,
    annotate_fmt_abs="{:,}",  # formatting for absolute labels
    save_path=None,
    bar_width=0.45
):
    # --- gather counters ---
    if rank not in per_rank_counters_a or rank not in per_rank_counters_b:
        raise ValueError(
            f"Rank '{rank}' not found in both inputs.\n"
            f"Available A: {list(per_rank_counters_a.keys())}\n"
            f"Available B: {list(per_rank_counters_b.keys())}"
        )

    counter_a = per_rank_counters_a[rank]
    counter_b = per_rank_counters_b[rank]

    taxa = set(counter_a.keys()) | set(counter_b.keys())
    if not taxa:
        raise ValueError("No taxa found for the given rank in either dataset.")

    df = pd.DataFrame({
        "taxon": list(taxa),
        label_a: [counter_a.get(t, 0) for t in taxa],
        label_b: [counter_b.get(t, 0) for t in taxa],
    })

    # drop all-zero taxa
    df = df[(df[label_a] > 0) | (df[label_b] > 0)]
    if df.empty:
        raise ValueError("All taxa have zero counts in both datasets for this rank.")

    # --- filter by RAW counts first ---
    if min_count is not None:
        df = df[(df[label_a] >= min_count) | (df[label_b] >= min_count)]
        if df.empty:
            raise ValueError("No taxa remain after applying min_count on raw counts.")

    # preserve raw counts for annotation later
    df["_raw_a"] = df[label_a].values
    df["_raw_b"] = df[label_b].values

    # --- normalize (optionally) ---
    y_label = "Occurrences"
    if normalize:
        sum_a = df["_raw_a"].sum()
        sum_b = df["_raw_b"].sum()
        df[label_a] = df["_raw_a"] / sum_a if sum_a > 0 else 0.0
        df[label_b] = df["_raw_b"] / sum_b if sum_b > 0 else 0.0
        y_label = "Relative share (%)"

        # optional post-normalization filter
        if min_prop is not None:
            df = df[(df[label_a] >= min_prop) | (df[label_b] >= min_prop)]
            if df.empty:
                raise ValueError("No taxa remain after applying min_prop on proportions.")

    # --- sort ---
    if sort_by == "a":
        df = df.sort_values(label_a, ascending=False)
    elif sort_by == "b":
        df = df.sort_values(label_b, ascending=False)
    else:  # "sum"
        df = df.assign(_sum=df[label_a] + df[label_b]).sort_values("_sum", ascending=False).drop(columns="_sum")

    # --- top-N ---
    if top_n is not None and len(df) > top_n:
        df = df.head(top_n)
    if df.empty:
        raise ValueError("No taxa remain after filtering/top_n.")

    # --- labels & colors ---
    wrapped_labels = ["\n".join(wrap(str(t), width=14)) for t in df["taxon"]]
    cmap = plt.get_cmap("tab20")
    color_a = cmap(0)
    color_b = cmap(12)

    # --- plot ---
    plt.figure(figsize=figsize)
    ax = plt.gca()
    x = range(len(df))

    bars_a = ax.bar([i - bar_width/2 for i in x], df[label_a].values, width=bar_width, label=label_a, color=color_a)
    bars_b = ax.bar([i + bar_width/2 for i in x], df[label_b].values, width=bar_width, label=label_b, color=color_b)

    if title is None:
        title = f"{y_label} by {rank.capitalize()} — {label_a} vs {label_b}"
    if normalize:
        ax.yaxis.set_major_formatter(ticker.PercentFormatter(1.0))
    ax.set_title(title)
    ax.set_xlabel(rank.capitalize())
    ax.set_ylabel(y_label)
    ax.set_xticks(list(x))
    ax.set_xticklabels(wrapped_labels, rotation=90, ha="right")
    ax.legend()
    plt.tight_layout()

    # --- annotate: absolute counts even when normalized ---
    if annotate:
        raw_a = df["_raw_a"].values
        raw_b = df["_raw_b"].values
        for rect, val_abs in zip(bars_a, raw_a):
            ax.text(
                rect.get_x() + rect.get_width()/2.0,
                rect.get_height(),
                annotate_fmt_abs.format(val_abs) if not normalize else annotate_fmt_abs.format(int(val_abs)),
                ha="center", va="bottom", rotation=90
            )
        for rect, val_abs in zip(bars_b, raw_b):
            ax.text(
                rect.get_x() + rect.get_width()/2.0,
                rect.get_height(),
                annotate_fmt_abs.format(val_abs) if not normalize else annotate_fmt_abs.format(int(val_abs)),
                ha="center", va="bottom", rotation=90
            )

    if save_path:
        plt.savefig(save_path)

    plt.show()


In [257]:
def update_sensors(df, feature_col="Light Spectra (Text) - verified", sensor_col="Imaging Method (Text) - verified"):
    def updater(features, sensors):
        sensors = list(sensors)  # copy to avoid mutating original

        # if len(sensors) > 1 and "Camera (manually triggered; e.g. Smartphone, System Camera, SLR Camera)" in sensors:
        #      sensors.remove("Camera (manually triggered; e.g. Smartphone, System Camera, SLR Camera)")

        if "Depth / LiDAR" in features:
            if "Depth Sensor / Lidar" not in sensors:
                sensors.append("Depth Sensor / Lidar")
                sensors.remove("Camera (manually triggered; e.g. Smartphone, System Camera, SLR Camera)")

        if "Depth / LiDAR" in features and ("RGB" in features or "RGB (visible)" in features):
            if "Camera (manually triggered; e.g. Smartphone, System Camera, SLR Camera)" not in sensors:
                sensors.append("Camera (manually triggered; e.g. Smartphone, System Camera, SLR Camera)")

        return sensors

    df[sensor_col] = df.apply(
        lambda row: updater(row[feature_col], row[sensor_col]),
        axis=1
    )
    return df

In [258]:
def fix_cv_tasks(tasks):
    res = []
    for task in tasks:
        if isinstance(task, str):
            y = task.lower()
            if y in ["identificaiton", "identification", "re-identification", "identifcation"]:
                res.append("Identification")
            elif y in ["counting", "detection", "localization"]:
                res.append("Detection")
            elif y in ["classification", "classifcation", "classificaiton"]:
                res.append("Classification")
            elif y in ["behaviour analysis", 'activity recognition', 'interaction monitoring', 'behavior analysis']:
                res.append("Activity Recognition")
            elif y in ["segmentation", "instance segmentation"]:
                res.append("Segmentation")
            else:
                res.append(task)
        else:
            res.append(task)
    if len(res) == 0:
        res.append("Unknown")
    return list(set(res))

In [286]:
def normalize_dataset_name(x):
    if isinstance(x, str):
        y = x.lower().strip() # Use lower() and strip() for better matching

        # --- Major Benchmarking Datasets ---

        # COCO (Common Objects in Context) - Placed high due to common substring "coco"
        if "coco" in y or "common objects in context" in y:
            return "COCO"

        # ImageNet / ILSVRC / Tiny ImageNet / mini-ImageNet
        elif "imagenet" in y or "ilsvrc" in y:
            return "ImageNet"

        # CIFAR variants (e.g., long-tailed)
        elif y.startswith("cifar") or "cifar-10" in y or "cifar-100" in y:
            return "CIFAR"

        # PASCAL VOC
        elif "pascal" in y or "voc" in y:
            return "PASCAL VOC"

        # ADE20K
        elif "ade20k" in y:
            return "ADE20K"

        # Stanford Cars / Cars-196
        elif "stanford car" in y or y in ["car", "cars", "car196", "cars196", "cars-196"]:
            return "Stanford Cars"

        # CUB (Caltech-UCSD Birds)
        elif y.startswith("cub") or "caltech-ucsd birds" in y:
            return "CUB-200-2011"

        # Oxford Flowers
        elif "oxford flowers" in y or "flower-102" in y or y == "flowers102":
            return "Oxford Flowers 102"

        # Labeled Faces in the Wild (LFW)
        elif "lfw" in y or "labeled faces in the wild" in y:
            return "LFW"

        # Human3.6M
        elif "human3.6m" in y or "h3.6m" in y or "h36m" in y:
            return "Human3.6M"

        # --- Specific Project and Thematic Datasets ---

        # NACTI (North American Camera Trap Images)
        elif "nacti" in y or "north american camera trap images" in y:
            return "NACTI"

        # IUCN Red List
        elif "iucn red list" in y:
            return "IUCN Red List"

        # GBIF (Global Biodiversity Information Facility)
        elif "gbif" in y or "global biodiversity information facility" in y:
            return "GBIF"

        # Animals with Attributes (AwA)
        elif "animals with attributes" in y or y.startswith("awa"):
            return "Animals with Attributes"

        # SAVMAP
        elif "savmap" in y:
            return "SAVMAP"

        # Happy Whale
        elif "happywhale" in y or "happy whale" in y:
            return "Happy Whale"

        # Rat7M
        elif "rat7m" in y or "rat 7m" in y:
            return "Rat7M"

        # MacaquePose
        elif "macaque" in y and "pose" in y:
            return "MacaquePose"

        # WildFish
        elif y.startswith("wildfish") or y.startswith("wild-fish"):
            return "WildFish"

        # FishBase
        elif "fishbase" in y:
            return "FishBase"

        # BANMo
        elif y.startswith("banmo"):
            return "BANMo"

        # Leeds Butterfly
        elif "leeds butterfly" in y:
            return "Leeds Butterfly"

        # Snapshot Serengeti
        elif "australia" in y and "snapshot" in y:
            return "Snapshot Australia"
        elif "wisconsin" in y and "snapshot" in y:
            return "Snapshot Wisconsin"
        elif "karoo" in y and "snapshot" in y:
            return "Snapshot Karoo"
        elif "enonkishu" in y and "snapshot" in y:
            return "Snapshot Enonkishu"
        elif "kgalagadi" in y and "snapshot" in y:
            return "Snapshot Kgalagadi"
        elif "camdeboo" in y and "snapshot" in y:
            return "Snapshot Camdeboo"
        elif "serengeti" in y or "snapshot" in y:  # fallback serengeti
            return "Snapshot Serengeti"

        # Google Open Images
        elif y.startswith("google open images") or y.startswith("open images dataset"):
            return "Open Images Dataset"

        # --- Existing and Other General Rules ---

        elif y in ["private", "unknown"]:
            return "Private"
        elif "wellington" in y:
            return "Wellington Camera Traps"
        elif y in ["rmas", "rsmas"]:
            return "RSMAS"
        elif y.startswith("lcf"):
            return "LCF"
        elif y in ["kaggle", "kaggle (unknown)"] or y.startswith("kaggle"):
            return "Kaggle"
        elif y.startswith("animal kingdom"):
            return "Animal Kingdom"
        elif "sealid" in y:
            return "SealID"
        elif y in ["kinetics-400", "kineteics-400"]:
            return "Kinetics-400"
        elif y in ["birds 525", "birds-525"]:
            return "Birds-525"
        elif y.startswith("wcs"):
            return "WCS Camera Traps"
        elif "megadetector" in y and "dataset" in y:
            return "MegaDetector dataset"
        elif "roboflow" in y or "robowflow" in y:
            return "Roboflow"
        elif y.startswith("inaturalist") or y.startswith("inat"):
            return "iNaturalist"
        elif y.startswith("stanford dog"):
            return "Stanford Dogs"
        elif y.startswith("oxford-iiit") or y.startswith("oxford pet"):
            return "Oxford-IIIT Pet dataset"
        elif y.startswith("fish4"):
            return "Fish4Knowledge"
        elif y.startswith("deepfish"):
            return "DeepFish"
        elif y.startswith("seaturtleid"):
            return "SeaTurtleID2022"
        elif y.startswith("fishnet"):
            return "FishNet"
        elif y.startswith("atrw"):
            return "ATRW"
        elif y.startswith("suim"):
            return "SUIM"
        elif y.startswith("wildlife2024"):
            return "WildLife2024"
        elif y.startswith("moca"):
            return "MOCA"
        elif y.startswith("pixabay"):
            return "Pixabay"
        elif y.startswith("google images"):
            return "Google Images"
        elif y.startswith("camo++"):
            return "CAMO++"
        elif y.startswith("caltech"):
            return "Caltech Camera Traps"
        elif y.startswith("dryad"):
            return "Dryad"
        elif y.startswith("zenodo"):
            return "Zenodo"
        elif y.startswith("figshare"):
            return "Figshare"
        elif y.startswith("lifeclef"):
            return "LifeCLEF"
        elif "wildcam" in y:
            return "iWildCam"
        elif "xeno-canto" in y or y == "xeno-canto":
            return "Xeno-canto"
        elif y.startswith("http:") or y.startswith("https://"):
            return None # "URL"
        elif "code" in y or "supplementary" in y or "data and analysis" in y:
             # Basic catch-all for code/supplemental files
            # if "dataset" not in y:
            #     return "Code/Supplemental"
            pass
    # If no rule matches, return the original name
    return x

In [260]:
# # gemini version
#
# import re
# from typing import Optional
#
# def normalize_dataset_name(x: any) -> Optional[str]:
#     """
#     Normalizes a dataset name using a series of rules and pattern matching.
#
#     This function cleans the input string and applies a prioritized list of rules
#     to map variations of dataset names to a single, canonical name.
#
#     The strategy involves:
#     1.  Robust pre-processing (lowercase, strip, remove punctuation).
#     2.  Filtering out generic/non-dataset entries first (e.g., "Source Code", "Supplementary Data").
#     3.  Matching specific, multi-word datasets before general ones.
#     4.  Grouping common benchmarks and thematic datasets from the semantic analysis.
#
#     Args:
#         x: The input dataset name, expected to be a string.
#
#     Returns:
#         A normalized string representing the canonical dataset name, or
#         None if the input is identified as a generic/non-dataset entry.
#         Returns the original input if it's not a string.
#     """
#     if not isinstance(x, str):
#         return x  # Return original if not a string
#
#     # 1. Aggressive Pre-processing
#     y = x.lower().strip()
#     # Replace common separators with spaces
#     y = y.replace('-', ' ').replace('_', ' ')
#     # Remove punctuation to simplify matching
#     y = re.sub(r'[^\w\s]', '', y)
#     # Collapse multiple spaces into one
#     y = re.sub(r'\s+', ' ', y).strip()
#
#     # 2. Filter out Generic/Supplemental/Code entries FIRST
#     # These are high-priority rules to remove non-dataset names.
#     generic_patterns = [
#         'supplement', 'supplemental', 'source data', 'raw data', 'supporting data',
#         'code', 'script', 'model', 'video', 'videos', 'images', 'all data',
#         'example', 'tutorial', 'repository', 'database', 'dataset', 'unaltered'
#     ]
#     # Regex for patterns like "Datafile S1", "S1 Dataset", "Figure 2", etc.
#     if re.search(r'^(datafile|dataset|figure|data)\s*s?\d+', y):
#         return None
#     if any(pattern in y for pattern in generic_patterns):
#         # A check to save actual named datasets that contain a generic word
#         # This is a heuristic and can be refined.
#         if 'dataset' in y and len(y.split()) > 2:
#             pass # Probably a real dataset name, let it be checked by later rules
#         else:
#             return None
#     if y.startswith("http") or y.endswith("com") or y.endswith("org"):
#         return None
#     if y in ["private", "unknown", "mixed", "general dataset", "complete dataset"]:
#         return "Private/Unavailable"
#
#
#     # 3. Major Benchmarking Datasets (High Specificity)
#     # Ordered to catch specific versions first (e.g., mini-ImageNet before ImageNet)
#     if 'coco' in y or 'common objects in context' in y:
#         return "COCO"
#     if 'imagenet' in y or 'ilsvrc' in y:
#         if 'mini' in y:
#             return "Mini-ImageNet"
#         if 'tiny' in y:
#             return "Tiny ImageNet"
#         return "ImageNet"
#     if 'cifar' in y:
#         return "CIFAR"
#     if 'pascal' in y or re.search(r'\bvoc\b', y):
#         return "PASCAL VOC"
#     if 'ade20k' in y:
#         return "ADE20K"
#     if 'lvis' in y:
#         return "LVIS"
#     if 'kinetics' in y:
#         if '400' in y: return "Kinetics-400"
#         if '700' in y: return "Kinetics-700"
#         return "Kinetics"
#     if 'places' in y:
#         return "Places"
#     if 'sun' in y:
#         return "SUN"
#
#
#     # 4. Human-centric Datasets (Faces, Pose, etc.)
#     if '300w' in y or '300 vw' in y:
#         return "300W"
#     if 'aflw' in y or 'annotated facial landmarks' in y:
#         return "AFLW"
#     if 'celeba' in y:
#         return "CelebA"
#     if 'wider face' in y:
#         return "WIDER Face"
#     if 'lfw' in y or 'labeled faces in the wild' in y:
#         return "LFW"
#     if 'human36m' in y or 'h36m' in y:
#         return "Human3.6M"
#     if 'macaquepose' in y:
#         return "MacaquePose"
#     if 'posetrack' in y:
#         return "PoseTrack"
#
#
#     # 5. Animal & Wildlife Datasets (from Semantic Analysis)
#     if 'atrw' in y or 'amur tiger re identification' in y:
#         return "ATRW"
#     if 'animal pose' in y:
#         return "Animal-Pose"
#     if 'serengeti' in y or y.startswith('snapshot'):
#         return "Snapshot Serengeti"
#     if 'nacti' in y or 'north american camera trap images' in y:
#         return "NACTI"
#     if 'animals with attributes' in y or y.startswith('awa'):
#         return "Animals with Attributes"
#     if 'smal' in y or 'skinned multi animal linear' in y:
#         return "SMAL"
#     if 'tigdog' in y:
#         return "TigDog"
#     if 'wild anim' in y:
#         return "Wild Animal Dataset"
#     if 'rat7m' in y or 'rat 7m' in y:
#         return "Rat7M"
#     if 'iwildcam' in y:
#         return "iWildCam"
#
#
#     # 6. Fish & Marine Life Datasets
#     if 'benthoz' in y:
#         return "Benthoz15"
#     if 'fin benthic' in y:
#         return "FIN-Benthic"
#     if 'ruie' in y or 'underwater image enhancement' in y:
#         return "RUIE"
#     if 'suim' in y or 'segmentation of underwater imagery' in y:
#         return "SUIM"
#     if 'uieb' in y:
#         return "UIEB"
#     if 'qut fish' in y or 'qutfish' in y:
#         return "QUT Fish Dataset"
#     if 'wildfish' in y or 'wild fish' in y:
#         return "WildFish"
#     if 'fishbase' in y:
#         return "FishBase"
#     if 'happywhale' in y or 'happy whale' in y:
#         return "Happy Whale"
#     if 'sealid' in y:
#         return "SealID"
#
#
#     # 7. Bird Datasets
#     if 'cub 200' in y or 'caltech ucsd birds' in y:
#         return "CUB-200-2011"
#     if 'fgvc aircraft' in y or y == 'air' or ('aircraft' in y and 'fgvc' in y):
#         return "FGVC-Aircraft"
#     if 'nabirds' in y:
#         return "NABirds"
#     if 'birdsnap' in y:
#         return "BirdSnap"
#
#
#     # 8. Insect Datasets
#     if 'aedes mosquitos' in y or 'aedes mosquitos' in y:
#         return "Aedes Mosquitos Dataset"
#     if 'leeds butterfly' in y:
#         return "Leeds Butterfly"
#
#     # 9. Vehicle & Scene Datasets
#     if 'stanford' in y and 'car' in y:
#         return "Stanford Cars"
#     if 'stanford' in y and 'dog' in y:
#         return "Stanford Dogs"
#     if 'oxford iiit pet' in y or 'oxford pet' in y:
#         return "Oxford-IIIT Pet"
#     if 'davis' in y and '2016' not in y and '2017' not in y: # Avoid matching DAVIS 2016/17
#         return "DAVIS"
#     if 'cityscape' in y:
#         return "Cityscapes"
#     if 'kitti' in y:
#         return "KITTI"
#     if 'ucsd' in y and 'pedestrian' in y:
#         return "UCSD Pedestrian"
#     if 'mall' in y:
#         return "Mall Dataset"
#     if 'change detection' in y or 'cdnet' in y:
#         return "CDnet"
#
#     # 10. General Repositories & Databases
#     if 'kaggle' in y:
#         return "Kaggle"
#     if 'inaturalist' in y or y.startswith('inat'):
#         return "iNaturalist"
#     if 'gbif' in y or 'global biodiversity information facility' in y:
#         return "GBIF"
#     if 'iucn red list' in y:
#         return "IUCN Red List"
#     if 'zenodo' in y:
#         return "Zenodo"
#     if 'figshare' in y:
#         return "Figshare"
#     if 'dryad' in y:
#         return "Dryad"
#     if 'lifeclef' in y or 'imageclef' in y:
#         return "LifeCLEF"
#     if 'xeno canto' in y:
#         return "Xeno-canto"
#     if 'google open image' in y or 'open images dataset' in y:
#         return "Open Images Dataset"
#     if 'google image' in y:
#         return "Google Images" # Less specific, so it comes after Open Images
#
#     # Return the original, unmodified name if no rule matches
#     return x

In [261]:
def unknown_light_spectra_to_other(dataframe, light_spectra_column):
    dataframe[light_spectra_column] = dataframe[light_spectra_column].apply(lambda x: [y if y != "Unknown" else "Other" for y in x])

In [262]:
manual_path = r"D:\LiteratureReviewCVinWC\review_output\manual.parquet"
auto_path = r"D:\LiteratureReviewCVinWC\review_output\20250731_fixed.parquet"
gbif_cache_path = r"D:\LiteratureReviewCVinWC\review_output\gbif_cache.json"

In [263]:
ma = convert_list_columns_to_sets(pd.read_parquet(manual_path))
auto = convert_list_columns_to_sets(pd.read_parquet(auto_path))

with open(gbif_cache_path, encoding="utf-8") as f:
    gbif_cache = json.load(f)

In [264]:
# ma["Light Spectra (Text) - verified"].explode().value_counts()

In [265]:
# ma = update_sensors(ma)

In [266]:
auto["doi"] = auto["doi"].apply(lambda x: urlparse(str(x)).path.lstrip("/"))
ma["doi"] = ma["doi"].apply(lambda x: urlparse(str(x)).path.lstrip("/"))


In [267]:
ma = ma[(ma["year"] >= 2014) & (ma["year"] <= 2024)]
mask = (
    ma["ParentHabitat values"].isna()
    | (ma["ParentHabitat values"] == "")
    | (ma["ParentHabitat values"].str.len() == 0)
)
ma.loc[mask, "ParentHabitat values"] = pd.Series(
    [["Unknown"] for _ in range(mask.sum())],
    index=ma.index[mask],
)


auto = auto[(auto["year"] >= 2014) & (auto["year"] <= 2024)]
mask = (
    auto["ParentHabitat values"].isna()
    | (auto["ParentHabitat values"] == "")
    | (auto["ParentHabitat values"].str.len() == 0)
)
auto.loc[mask, "ParentHabitat values"] = pd.Series(
    [["Unknown"] for _ in range(mask.sum())],
    index=auto.index[mask],
)


In [268]:
common = ma[ma["doi"].isin(auto["doi"])]
ma = ma[ma["doi"].isin(common["doi"])]
auto = auto[auto["doi"].isin(common["doi"])]

In [269]:
ma['ParentHabitat values'] = ma['ParentHabitat values'].apply(map_all_to_first_level)
auto['ParentHabitat values'] = auto['ParentHabitat values'].apply(map_all_to_first_level)

In [270]:
auto['CV Tasks - verified'] = auto['CV Tasks - verified'].map(fix_cv_tasks)
ma['CV Tasks - verified'] = ma['CV Tasks - verified'].map(fix_cv_tasks)

In [271]:
# auto["Imaging Method (new) - verified"].explode().value_counts()

In [272]:
auto["Imaging Method (new) - verified"] = auto['Imaging Method (new) - verified'].apply(unify_imaging_methods)
ma["Imaging Method (Text) - verified"] = ma['Imaging Method (Text) - verified'].apply(unify_imaging_methods)


In [273]:
def remove_row_with_value(df, column, value):
    # Step 1: Drop rows where the list only contains the value
    df = df[df[column].apply(lambda x: not (len(x) == 1 and x[0] == value))].copy()

    # Step 2: Remove the value from lists with other values
    df.loc[:, column] = df[column].apply(lambda x: [m for m in x if m != value])
    return df

ma = remove_row_with_value(ma, "Imaging Method (Text) - verified", "Acoustic Camera")
auto = remove_row_with_value(auto, "Imaging Method (new) - verified", "Acoustic Camera")

ma = remove_row_with_value(ma, "Light Spectra (Text) - verified", "Acoustic")
auto = remove_row_with_value(auto, "Light Spectra (Text) (new) - verified", "Acoustic")
auto = remove_row_with_value(auto, "Light Spectra (Text) (new) - verified", "Active Imaging Sonar (multibeam)")
unknown_light_spectra_to_other(ma, "Light Spectra (Text) - verified")
unknown_light_spectra_to_other(auto, "Light Spectra (Text) (new) - verified")


In [274]:
ma#[ma["doi"] == "https://doi.org/10.1016/j.suscom.2023.100858"]

,file,doi,year,Species (Text)(translated) - verified,Species (Text)(translated) - unverified,Species (Images)(translated) - verified,Species (Images)(translated) - unverified,Country - verified,Country - unverified,Imaging Method - verified,...,Imaging Method (Text) - verified,Imaging Method (Text) - unverified,CV Tasks - verified,CV Tasks - unverified,CV Algorithms - verified,CV Algorithms - unverified,Dataset - verified,Dataset - unverified,HabitatVerification values,ParentHabitat values
0,10.1002ajp.22627.json,10.1002/ajp.22627,2017,[Pan troglodytes],[],[],[],"[Liberia, Uganda]",[],[],...,[Camera Trap],[],[Identification],[],[SHORE™],[],[Private],[],[],[Forest]
1,10.1002cbdv.202201123.json,10.1002/cbdv.202201123,2023,"[Mullus barbatus, Sparus aurata, Salmo trutta,...",[],[],[],[Turkey],[],[],...,[Non-specialized Camera],[],[Classification],[],"[Xception, CNN, Inception, DenseNet, VGG, ResNet]",[],[https://doi.org/10.1109/ASYU50717.2020.9259867],[],[],"[Artificial Aquatic, Artificial Terrestrial, W..."
2,10.1002eap.2694.json,10.1002/eap.2694,2022,"[Ardea, Pelecanus, Anatidae, Grus, Anser, Laru...",[],[],[],"[Poland, Antarctica, Guinea, USA]",[],[],...,[Unmanned aerial vehicle (UAV; e.g. drone)],[],"[Tracking, Detection]",[],[RetinaNet],[],[https://doi.org/10.5281/zenodo.5033174],[],[],"[Wetlands (inland), Forest, Marine]"
3,10.1002ece3.4747.json,10.1002/ece3.4747,2019,[Animalia],[],[],[],[USA],[],[],...,[Camera Trap],[],[Detection],[],"[Faster R-CNN, BOW, AlexNet, SSD, RRC]",[],[Private],[],[],[Forest]
4,10.1002ece3.5695.json,10.1002/ece3.5695,2019,[Aves],[],[],[],[USA],[],[],...,[Non-specialized Camera],[],[Detection],[],"[LBP, HOG, Haar cascade]",[],[Private],[],[],"[Artificial Terrestrial, Grassland]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
502,10.3758s13428-020-01381-9.json,10.3758/s13428-020-01381-9,2020,[Rattus norvegicus],[],[],[],[Portugal],[],[],...,"[Depth Sensor / Lidar, Non-specialized Camera]",[],"[Tracking, Classification]",[],"[SVM, landscape change detection, GMM]",[],[https://doi.org/10.5281/zenodo.3636135],[],[],"[Artificial Aquatic, Artificial Terrestrial]"
510,10.7554elife.48571.json,10.7554/elife.48571,2019,[Drosophila],[],[],[],[Canada],[],[],...,[Non-specialized Camera],[],[Pose Estimation],[],[Stacked Hourglass human pose estimation network],[],[https://github.com/NeLy-EPFL/DeepFly3D],[],[],"[Artificial Aquatic, Artificial Terrestrial]"
511,10.7554elife.58145.json,10.7554/elife.58145,2020,"[Camponotus fellah, Camponotus, Ooceraea biroi...",[],[],[],[USA],[],[],...,[Non-specialized Camera],[],"[Tracking, Detection]",[],"[Optical Flow, CNN]",[],[https://doi.org/10.5281/zenodo.3740547],[],[],"[Artificial Aquatic, Artificial Terrestrial]"
514,10.7717peerj-cs.1250.json,10.7717/peerj-cs.1250,2023,[Canis familiaris],[],[],[],[Unknown],[],[],...,[Non-specialized Camera],[],[Classification],[],[GAN],[],[COCO],[],[],[Unknown / Other]


In [275]:
auto

,file,doi,year,Species (Text)(translated) - verified,Species (Text)(translated) - unverified,Species (Images)(translated) - verified,Species (Images)(translated) - unverified,Country - verified,Country - unverified,Imaging Method (new) - verified,...,Light Spectra (Images) (new) - verified,Light Spectra (Images) (new) - unverified,CV Tasks - verified,CV Tasks - unverified,CV Algorithms - verified,CV Algorithms - unverified,Dataset - verified,Dataset - unverified,HabitatVerification values,ParentHabitat values
2,10.1002ajp.22627.json,10.1002/ajp.22627,2017,"[Gorilla, Pan troglodytes schweinfurthii]",[],[Pan troglodytes schweinfurthii],[],"[Liberia, Uganda]",[],[Camera Trap],...,"[RGB (visible), Grayscale]",[],"[Tracking, Identification, Classification, Det...",[],"[structure features, Sobel operators, mean fil...",[],[private],[],[Forest.forest_subtropical_tropical_moist_lowl...,"[Artificial Terrestrial, Forest]"
5,10.1002cbdv.202201123.json,10.1002/cbdv.202201123,2023,"[Pagellus bogaraveo, Sparus aurata, Salmo trut...",[Mullus surmuletus],"[Pagellus bogaraveo, Sparus aurata, Salmo trut...",[],[Turkey],[],[Non-specialized Camera],...,[],"[RGB (visible), Grayscale]","[Classification, Segmentation]",[],"[VGG-19, Chaotic Oppositional Based Whale Opti...","[Xception, Inception V3, ResNet150V2, DenseNet]",[private],[],[Unknown.unknown_habitat],"[Unknown / Other, Wetlands (inland)]"
8,10.1002eap.2694.json,10.1002/eap.2694,2022,"[Eudocimus albus, Egretta thula, Platalea ajaj...",[],"[Pygoscelis antarcticus, Anatidae, Sternidae, ...",[Spheniscidae],"[Canada, Antarctica (C), Guinea, Falkland Isla...",[],"[Unmanned aerial vehicle (UAV; e.g. drone), Ot...",...,[RGB (visible)],[],[Detection],[],"[focal loss, ResNet-50 network, RetinaNet dete...",[],"[Weinstein, Garner, et al., 2021]",[],[WetlandsInland.wetlands_inland_bogs_marshes_s...,"[Grassland, Desert, Shrubland, Marine, Wetland..."
17,10.1002ece3.4747.json,10.1002/ece3.4747,2019,[],[],"[Lynx rufus, Bovidae, Galliformes, Procyon lot...",[Odocoileus],"[Tanzania, Panama, Netherlands, USA]",[],[Camera Trap],...,"[RGB (visible), Grayscale]",[],"[Classification, Detection, Segmentation]",[],"[Shrinked Histogram Length, Histogram of Orien...",[AlexNet-96],"[Snapshot Serengeti, private]",[],"[Forest.forest_temperate, Shrubland.shrubland_...","[Grassland, Desert, Unknown / Other, Shrubland..."
19,10.1002ece3.5695.json,10.1002/ece3.5695,2019,"[Molothrus ater, Zenaida macroura]",[],"[Molothrus ater, Zenaida macroura]",[],[USA],[],[Non-specialized Camera],...,[RGB (visible)],[],"[Classification, Detection]",[],"[local binary patterns, histogram of oriented ...",[],[William & Mary Applied Science Department Non...,[],"[ArtificialTerrestrial.urban_areas, Artificial...","[Artificial Terrestrial, Forest, Grassland]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1756,10.3758s13428-020-01381-9.json,10.3758/s13428-020-01381-9,2020,[Rattus norvegicus],[],"[Rattus norvegicus, Rodentia]",[],"[Finland, Portugal]",[],[Depth Sensor / Lidar],...,"[RGB (visible), Depth / LiDAR]",[],"[Pose Estimation, Classification, Detection, T...",[],"[support vector machine (SVM), static median d...",[],"[private, RGB-D rat dataset]",[],[],[Unknown / Other]
1773,10.7554eLife.48571.json,10.7554/elife.48571,2019,"[Drosophila melanogaster, Homo sapiens]",[],[Drosophila],[],"[Switzerland, Romania, Canada]",[],[Non-specialized Camera],...,[Grayscale],[Red Band],"[Pose Estimation, Tracking, Activity Recogniti...",[],"[Pictorial Structures, t-SNE, Max-Sum algorith...",[1€ filter],"[aDN-GAL4 Control, MDN-GAL4 Control, MDN-GAL4 ...",[],[],[Unknown / Other]
1775,10.7554elife.58145.json,10.7554/elife.58145,2020,"[Camponotus fellah, Camponotus, Ooceraea biroi...",[],"[Camponotus fellah, Camponotus, Ooceraea biroi...",[],[United States],[],[Non-specialized Camera],...,"[RGB (visible), Grayscale]",[],"[Pose Estimation, Classification, Detection, T...",[],"[morphological operations, thresholding, c

In [292]:
def simplify_cv_tasks(values):
    to_remove = []
    if "Tracking" in values and "Detection" in values:
        to_remove.append("Detection")
    elif "Tracking" in values and "Identification" in values:
        to_remove.append("Identification")
    elif "Detection" in values and "Classification" in values:
        to_remove.append("Classification")
    res = values.copy()
    for r in to_remove:
        res.remove(r)
    return res

In [297]:
ma["CV Tasks - verified"] = ma["CV Tasks - verified"].apply(simplify_cv_tasks)
auto["CV Tasks - verified"] = auto["CV Tasks - verified"].apply(simplify_cv_tasks)

In [298]:
merged = pd.merge(ma, auto, on='doi', how='outer', suffixes=('_ma', '_auto'))

In [299]:
merged[["doi", "CV Tasks - verified_ma", "CV Tasks - verified_auto"]]

,doi,CV Tasks - verified_ma,CV Tasks - verified_auto
0,10.1002/ajp.22627,[Identification],"[Tracking, Classification]"
1,10.1002/cbdv.202201123,[Classification],"[Classification, Segmentation]"
2,10.1002/eap.2694,[Tracking],[Detection]
3,10.1002/ece3.4747,[Detection],"[Detection, Segmentation]"
4,10.1002/ece3.5695,[Detection],[Detection]
...,...,...,...
186,10.3758/s13428-020-01381-9,"[Tracking, Classification]","[Pose Estimation, Classification, Tracking, Ac..."
187,10.7554/elife.48571,[Pose Estimation],"[Pose Estimation, Tracking, Activity Recogniti..."
188,10.7554/elife.58145,[Tracking],"[Pose Estimation, Classification, Tracking, Ac..."
189,10.7717/peerj-cs.1250,[Classification],"[Synthesis, Classification]"


In [300]:
def merge_list_columns(df: pd.DataFrame, cols_to_merge: list, new_col: str) -> pd.DataFrame:
    """
    Merges multiple list-based columns into one list column and removes the originals.

    Parameters
    ----------
    df : pd.DataFrame
        The input DataFrame.
    cols_to_merge : list
        List of column names whose lists should be merged.
    new_col : str
        Name of the new column that will contain the merged lists.

    Returns
    -------
    pd.DataFrame
        DataFrame with the new merged column and without the original columns.
    """

    def safe_merge(row):
        merged = []
        for col in cols_to_merge:
            val = row.get(col, [])
            if isinstance(val, list):
                merged.extend(val)
            elif pd.notna(val):
                merged.append(val)
        return merged

    df[new_col] = df.apply(safe_merge, axis=1)
    return df.drop(columns=cols_to_merge)

In [301]:
# list_col_map = {
#     "Species (Text)(translated) - verified": "Species (Text)(translated) - verified",
#     "Species (Images)(translated) - verified": "Species (Images)(translated) - verified",
#     "Country - verified": "Country - verified",
#     "Imaging Method (Text) - verified": "Imaging Method (new) - verified",
#     "Light Spectra (Text) - verified": "Light Spectra (Text) (new) - verified",
#     "Light Spectra (Images) (new) - verified": "Light Spectra (Images) (new) - verified",
#     "CV Tasks - verified": "CV Algorithms - verified",
#     "Dataset - verified": "Dataset - verified",
#     "ParentHabitat values": "ParentHabitat values"
# }

ma2 = merge_list_columns(ma, ["Species (Text)(translated) - verified", "Species (Images)(translated) - verified"], "Species")
ma2 = merge_list_columns(ma2, ["Light Spectra (Text) - verified", "Light Spectra (Images) - verified"], "Light Spectra")

auto2 = merge_list_columns(auto, ["Species (Text)(translated) - verified", "Species (Images)(translated) - verified"], "Species")
auto2 = merge_list_columns(auto2, ["Light Spectra (Text) (new) - verified", "Light Spectra (Images) (new) - verified"], "Light Spectra")


In [302]:
from typing import List, Dict, Any
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment

def embed_texts(texts: List[str], model_name: str = "sentence-transformers/all-MiniLM-L6-v2") -> np.ndarray:
    model = SentenceTransformer(model_name)
    return model.encode(texts, normalize_embeddings=True)

def pairwise_cosine_sim(a_emb: np.ndarray, b_emb: np.ndarray) -> np.ndarray:
    return cosine_similarity(a_emb, b_emb)  # values in [-1, 1], higher = more similar

def greedy_best_match(sim: np.ndarray) -> Dict[str, Any]:
    # For each row (A item) pick the best B match. Also do the reverse for symmetry.
    a2b_idx = sim.argmax(axis=1)
    a2b_scores = sim[np.arange(sim.shape[0]), a2b_idx]

    b2a_idx = sim.argmax(axis=0)
    b2a_scores = sim[b2a_idx, np.arange(sim.shape[1])]

    return {
        "a_to_b_indices": a2b_idx.tolist(),
        "a_to_b_scores": a2b_scores.tolist(),
        "a_to_b_mean": float(a2b_scores.mean()) if a2b_scores.size else None,
        "b_to_a_indices": b2a_idx.tolist(),
        "b_to_a_scores": b2a_scores.tolist(),
        "b_to_a_mean": float(b2a_scores.mean()) if b2a_scores.size else None,
    }

def hungarian_one_to_one(sim: np.ndarray) -> Dict[str, Any]:
    # We want to maximize similarity, Hungarian minimizes cost, so use negative similarity.
    # If matrix is not square, pad with zeros (or a small value) to allow a full assignment.
    n_rows, n_cols = sim.shape
    n = max(n_rows, n_cols)
    padded = np.zeros((n, n))
    padded[:n_rows, :n_cols] = sim

    row_ind, col_ind = linear_sum_assignment(-padded)  # maximize sim
    # Filter out assignments that are in the padded area
    valid = [(r, c) for r, c in zip(row_ind, col_ind) if r < n_rows and c < n_cols]
    scores = [sim[r, c] for r, c in valid]

    return {
        "matches": valid,            # list of (i_in_A, j_in_B)
        "scores": scores,
        "mean_score": float(np.mean(scores)) if scores else None,
        "coverage_A": len({r for r,_ in valid}) / n_rows if n_rows else None,
        "coverage_B": len({c for _,c in valid}) / n_cols if n_cols else None,
    }

def list_similarity(
    list_a: List[str],
    list_b: List[str],
    model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
    strategy: str = "hungarian"  # "hungarian" or "greedy"
) -> Dict[str, Any]:
    a_emb = embed_texts(list_a, model_name)
    b_emb = embed_texts(list_b, model_name)
    sim = pairwise_cosine_sim(a_emb, b_emb)

    if strategy == "hungarian":
        summary = hungarian_one_to_one(sim)
    elif strategy == "greedy":
        summary = greedy_best_match(sim)
    else:
        raise ValueError("strategy must be 'hungarian' or 'greedy'")

    return {
        "model": model_name,
        "strategy": strategy,
        "pairwise_similarity_matrix": sim,  # numpy array
        "summary": summary
    }

def _as_list_of_str(x):
    if isinstance(x, float) and np.isnan(x):
        return []
    if x is None:
        return []
    if isinstance(x, str):
        return [x.strip()] if x.strip() else []
    if isinstance(x, (list, tuple, set)):
        return [str(t).strip() for t in x if str(t).strip()]
    # fallback
    return [str(x)]

In [303]:
def compare_dataframes_by_doi(
    df_left: pd.DataFrame,
    df_right: pd.DataFrame,
    string_col_map: dict,
    list_col_map: dict,
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    strategy="hungarian"
):
    """
    Compare two DataFrames row-wise, matched on DOI.

    Parameters
    ----------
    df_left, df_right : pd.DataFrame
        Input dataframes. Both must have a 'doi' column.
    string_col_map : dict
        Mapping { "left_col" : "right_col" } for string columns.
    list_col_map : dict
        Mapping { "left_col" : "right_col" } for list-of-strings columns.
    """

    # Keep only rows with matching DOIs
    common = set(df_left["doi"]).intersection(set(df_right["doi"]))
    left_ix  = df_left[df_left["doi"].isin(common)].set_index("doi")
    right_ix = df_right[df_right["doi"].isin(common)].set_index("doi")

    results = []
    detailed = {}

    for doi in common:
        rowL = left_ix.loc[doi]
        rowR = right_ix.loc[doi]

        col_scores = {}
        col_details = {}

        # --- string columns ---
        for lcol, rcol in string_col_map.items():
            a = _as_list_of_str(rowL.get(lcol, None))
            b = _as_list_of_str(rowR.get(rcol, None))
            if len(a) == 0:
                a.append("")
            if len(b) == 0:
                b.append("")
            if len(a) == 0 and len(b) == 0:
                score = None
                detail = {"note": "both empty"}
            else:
                out = list_similarity(a, b, model_name=model_name, strategy=strategy)
                score = out["summary"]["mean_score"] if strategy == "hungarian" else out["summary"]["a_to_b_mean"]
                detail = out["summary"]

            col_scores[f"{lcol}_vs_{rcol}"] = score
            col_details[f"{lcol}_vs_{rcol}"] = detail

        # --- list-of-strings columns ---
        for lcol, rcol in list_col_map.items():
            a = _as_list_of_str(rowL.get(lcol, []))
            b = _as_list_of_str(rowR.get(rcol, []))
            if len(a) == 0:
                a.append("")
            if len(b) == 0:
                b.append("")
            if len(a) == 0 and len(b) == 0:
                score = None
                detail = {"note": "both empty"}
            else:
                out = list_similarity(a, b, model_name=model_name, strategy=strategy)
                score = out["summary"]["mean_score"] if strategy == "hungarian" else out["summary"]["a_to_b_mean"]
                detail = out["summary"]

            col_scores[f"{lcol}_vs_{rcol}"] = score
            col_details[f"{lcol}_vs_{rcol}"] = detail

        # aggregate score
        sim_values = [v for v in col_scores.values() if v is not None]
        overall = float(np.mean(sim_values)) if sim_values else None

        results.append({"doi": doi, "overall_similarity": overall, **col_scores})
        detailed[doi] = col_details

    summary_df = pd.DataFrame(results).set_index("doi").sort_values("overall_similarity", ascending=False)
    return summary_df, detailed

In [304]:
import os, json, pickle, tempfile, time
from typing import Dict, Tuple
import pandas as pd

# Requires: pyarrow (recommended) for Parquet I/O
# pip install pyarrow

def _settings_key(string_col_map: dict, list_col_map: dict, model_name: str, strategy: str) -> str:
    """Stable key for the overall settings; stored alongside each row in the Parquet."""
    import hashlib
    payload = json.dumps({
        "string_col_map": dict(sorted(string_col_map.items())),
        "list_col_map":   dict(sorted(list_col_map.items())),
        "model_name": model_name,
        "strategy": strategy
    }, sort_keys=True).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()[:16]

def _as_json(obj) -> str:
    return json.dumps(obj, ensure_ascii=False, sort_keys=True)

def _load_cache_parquet(cache_path: str) -> pd.DataFrame:
    if os.path.exists(cache_path):
        return pd.read_parquet(cache_path)
    # define a stable schema even if empty
    return pd.DataFrame(columns=[
        "doi", "overall_similarity", "settings_key", "updated_at", "details_json"
        # plus dynamic per-column similarity columns that will appear as we write
    ])

def _atomic_write_parquet(df: pd.DataFrame, cache_path: str):
    """Write Parquet atomically (tmp + replace) to avoid corruption on interrupts."""
    os.makedirs(os.path.dirname(cache_path) or ".", exist_ok=True)
    with tempfile.NamedTemporaryFile("wb", delete=False, dir=os.path.dirname(cache_path) or ".") as tmp:
        tmp_path = tmp.name
    try:
        df.to_parquet(tmp_path, index=False)
        os.replace(tmp_path, cache_path)
    finally:
        if os.path.exists(tmp_path):
            try:
                os.remove(tmp_path)
            except OSError:
                pass

import numpy as np
import pandas as pd
import json

def _to_python(obj):
    """Recursively convert numpy/pandas types to plain Python for JSON/parquet safety."""
    if isinstance(obj, dict):
        return {str(k): _to_python(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [_to_python(v) for v in obj]
    if isinstance(obj, (np.generic,)):          # any numpy scalar
        return obj.item()
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    # Pandas NA/NaT -> None
    if obj is pd.NA or obj is None:
        return None
    try:
        # catches pandas Timestamp/Timedelta
        if isinstance(obj, (pd.Timestamp, pd.Timedelta)):
            return obj.isoformat()
    except Exception:
        pass
    # Convert pandas nullable scalars (e.g., pd.Int64Dtype()) to Python
    if hasattr(obj, "__class__") and obj.__class__.__name__ in {"IntegerArray", "FloatingArray"}:
        return obj.astype(object).tolist()
    # Leave plain Python types as is
    return obj

from pandas.api.types import (
    is_float_dtype, is_integer_dtype, is_bool_dtype, is_extension_array_dtype
)

def missing_for(dtype):
    # Pandas nullable dtypes (Int64, Float64, boolean, string, Arrow, etc.) accept pd.NA
    if is_extension_array_dtype(dtype):
        return pd.NA
    # Plain NumPy floats use np.nan
    if is_float_dtype(dtype):
        return np.nan
    # Plain NumPy ints/bools can't hold NA; either use np.nan (will upcast) or
    # convert column to nullable first (shown below).
    if is_integer_dtype(dtype) or is_bool_dtype(dtype):
        return np.nan
    # objects/strings etc.
    return pd.NA

def compare_dataframes_by_doi_incremental_parquet(
    df_left: pd.DataFrame,
    df_right: pd.DataFrame,
    string_col_map: Dict[str, str],
    list_col_map: Dict[str, str],
    *,
    model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
    strategy: str = "hungarian",
    cache_path: str = r"D:\LiteratureReviewCVinWC\review_output\similarity_cache.parquet",
    force_refresh: bool = False,
    doi_col: str = "doi"
) -> Tuple[pd.DataFrame, Dict[str, dict]]:
    """
    Incrementally computes similarity per DOI and persists to ONE Parquet file after each DOI.
    - Skips DOIs already in the cache (same settings_key), unless force_refresh=True.
    - Stores a JSON string per row with detailed matching info (per column).
    - Returns the in-memory summary DataFrame (current run’s DOIs) and a dict of details.
    """
    settings = _settings_key(string_col_map, list_col_map, model_name, strategy)

    # Build indices only for DOIs present on both sides
    common = set(df_left[doi_col]).intersection(set(df_right[doi_col]))
    left_ix  = df_left[df_left[doi_col].isin(common)].set_index(doi_col)
    right_ix = df_right[df_right[doi_col].isin(common)].set_index(doi_col)

    # Load or create cache
    cache_df = _load_cache_parquet(cache_path)

    # Restrict the "already done" set to rows with the same settings_key
    done_mask = (cache_df["settings_key"] == settings) if not cache_df.empty else pd.Series([], dtype=bool)
    done_dois = set(cache_df.loc[done_mask, "doi"].astype(str))

    results = []
    detailed_all = {}

    length = len(common)
    for doi_idx, doi in enumerate(common):
        print(f"{doi_idx+1}/{length}: Processing {doi}...")
        doi_str = str(doi)

        # Skip if cached (same settings) unless forced
        if not force_refresh and doi_str in done_dois:
            # collect from cache for return
            row = cache_df.loc[(cache_df["settings_key"] == settings) & (cache_df["doi"] == doi_str)]
            if not row.empty:
                print("--- Reading from cache")
                # reconstruct minimal "detailed" dict from json (optional)
                try:
                    detailed_all[doi_str] = json.loads(row.iloc[0]["details_json"]) if "details_json" in row else {}
                except Exception:
                    detailed_all[doi_str] = {}
                # also add to results for this call
                results.append(row.iloc[0].drop(labels=["settings_key", "updated_at", "details_json"], errors="ignore").to_dict())
            continue

        # --- compute fresh for this DOI ---
        rowL = left_ix.loc[doi]
        rowR = right_ix.loc[doi]

        col_scores = {}
        col_details = {}

        # string columns
        for lcol, rcol in string_col_map.items():
            a = _as_list_of_str(rowL.get(lcol, None))
            b = _as_list_of_str(rowR.get(rcol, None))
            if len(a) == 0:
                a.append("")
            if len(b) == 0:
                b.append("")
            if len(a) == 0 and len(b) == 0:
                score = None
                detail = {"note": "both empty"}
            else:
                out = list_similarity(a, b, model_name=model_name, strategy=strategy)
                score = out["summary"]["mean_score"] if strategy == "hungarian" else out["summary"]["a_to_b_mean"]
                detail = out["summary"]
            key = f"{lcol}_vs_{rcol}"
            col_scores[key] = score
            col_details[key] = detail

        # list-of-strings columns
        for lcol, rcol in list_col_map.items():
            a = _as_list_of_str(rowL.get(lcol, []))
            b = _as_list_of_str(rowR.get(rcol, []))
            if len(a) == 0:
                a.append("")
            if len(b) == 0:
                b.append("")
            if len(a) == 0 and len(b) == 0:
                score = None
                detail = {"note": "both empty"}
            else:
                out = list_similarity(a, b, model_name=model_name, strategy=strategy)
                score = out["summary"]["mean_score"] if strategy == "hungarian" else out["summary"]["a_to_b_mean"]
                detail = out["summary"]
            key = f"{lcol}_vs_{rcol}"
            col_scores[key] = score
            col_details[key] = detail

        # overall score
        sim_values = [v for v in col_scores.values() if v is not None]
        overall = float(sum(sim_values)/len(sim_values)) if sim_values else None

        col_details_py = _to_python(col_details)
        # prepare a single-row DF for appending/upserting
        row_out = {
            "doi": doi_str,
            "overall_similarity": overall,
            "settings_key": settings,
            "updated_at": pd.Timestamp.utcnow().isoformat(),
            "details_json": json.dumps(col_details_py, ensure_ascii=False, sort_keys=True)
        }
        for k, v in col_scores.items():
            row_out[k] = float(v) if isinstance(v, (np.floating, float)) and v is not None else (
                int(v) if isinstance(v, (np.integer, int)) and v is not None else (None if v is None else _to_python(v))
            )
        row_out.update(col_scores)

        # --- upsert this DOI into the cache df ---
        if cache_df.empty:
            cache_df = pd.DataFrame([row_out])
        else:
            # if DOI with same settings exists -> replace; else append
            mask = (cache_df["settings_key"] == settings) & (cache_df["doi"] == doi_str)
            if mask.any():
                # align columns if new columns appeared
                for c in row_out.keys():
                    if c not in cache_df.columns:
                        cache_df[c] = pd.NA
                # also ensure cache_df doesn't miss columns from earlier rows when writing new ones
                for c in cache_df.columns:
                    if c not in row_out:
                        row_out[c] = missing_for(cache_df[c].dtype)
                cache_df.loc[mask, list(row_out.keys())] = pd.Series(row_out)
            else:
                # align columns
                for c in row_out.keys():
                    if c not in cache_df.columns:
                        cache_df[c] = pd.NA
                cache_df = pd.concat([cache_df, pd.DataFrame([row_out])], ignore_index=True)

        # write Parquet atomically after EVERY DOI
        _atomic_write_parquet(cache_df, cache_path)

        # add to current-run outputs
        results.append({k: v for k, v in row_out.items() if k not in ("settings_key", "updated_at", "details_json")})
        detailed_all[doi_str] = col_details

    # Build a summary_df (only current-run DOIs; if you want full cache, read cache_path)
    summary_df = pd.DataFrame(results).set_index("doi").sort_values("overall_similarity", ascending=False)
    return summary_df, detailed_all


In [313]:
def dataset_to_private_or_public(items):
    res = set()
    for x in items:
        if isinstance(x, str):
            y = x.lower().strip()
            if y in ["private", "unknown"]:
                res.add("Private")
            else:
                res.add("Public")
        else:
            res.add("Public")
    if len(res) == 0:
        res.add("Private")
    return list(res)
ma2["Dataset Clustered - verified"] = ma2["Dataset - verified"].map(dataset_to_private_or_public)
auto2["Dataset Clustered - verified"] = auto2["Dataset - verified"].map(dataset_to_private_or_public)

In [314]:
string_col_map = {
}
# list_col_map = {
#     "Species (Text)(translated) - verified": "Species (Text)(translated) - verified",
#     "Species (Images)(translated) - verified": "Species (Images)(translated) - verified",
#     "Country - verified": "Country - verified",
#     "Imaging Method (Text) - verified": "Imaging Method (new) - verified",
#     "Light Spectra (Text) - verified": "Light Spectra (Text) (new) - verified",
#     "Light Spectra (Images) (new) - verified": "Light Spectra (Images) (new) - verified",
#     "CV Tasks - verified": "CV Algorithms - verified",
#     "Dataset - verified": "Dataset - verified",
#     "ParentHabitat values": "ParentHabitat values"
# }

print(ma2.columns.tolist())
print(auto2.columns.tolist())

list_col_map = {
    "Species": "Species",
    "Country - verified": "Country - verified",
    "Imaging Method (Text) - verified": "Imaging Method (new) - verified",
    "Light Spectra": "Light Spectra",
    "CV Tasks - verified": "CV Tasks - verified",
    "CV Algorithms - verified": "CV Algorithms - verified",
    "Dataset - verified": "Dataset - verified",
    "Dataset Clustered - verified": "Dataset Clustered - verified",
    "ParentHabitat values": "ParentHabitat values"
}

# index = 2
# summary_df, detailed = compare_dataframes_by_doi_incremental_parquet(
#     ma.iloc[[index]], auto.iloc[[index]],
#     string_col_map=string_col_map,
#     list_col_map=list_col_map
# )
summary_df, detailed = compare_dataframes_by_doi_incremental_parquet(
    ma2, auto2,
    string_col_map=string_col_map,
    list_col_map=list_col_map,
    force_refresh=True
)

# print(summary_df.head())

['file', 'doi', 'year', 'Species (Text)(translated) - unverified', 'Species (Images)(translated) - unverified', 'Country - verified', 'Country - unverified', 'Imaging Method - verified', 'Imaging Method - unverified', 'Light Spectra (Text) - unverified', 'Light Spectra (Images) - unverified', 'Imaging Method (Text) - verified', 'Imaging Method (Text) - unverified', 'CV Tasks - verified', 'CV Tasks - unverified', 'CV Algorithms - verified', 'CV Algorithms - unverified', 'Dataset - verified', 'Dataset - unverified', 'HabitatVerification values', 'ParentHabitat values', 'Species', 'Light Spectra', 'Dataset Clustered - verified']
['file', 'doi', 'year', 'Species (Text)(translated) - unverified', 'Species (Images)(translated) - unverified', 'Country - verified', 'Country - unverified', 'Imaging Method (new) - verified', 'Imaging Method (new) - unverified', 'Light Spectra (Text) (new) - unverified', 'Light Spectra (Images) (new) - unverified', 'CV Tasks - verified', 'CV Tasks - unverified', 

C:\Users\P41743\AppData\Local\Temp\ipykernel_8540\3886511272.py:233: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  cache_df = pd.concat([cache_df, pd.DataFrame([row_out])], ignore_index=True)


2/189: Processing 10.1016/j.ecss.2022.107815...
3/189: Processing 10.1038/s41598-022-21910-0...
4/189: Processing 10.1049/ipr2.13027...
5/189: Processing 10.1242/jeb.115063...
6/189: Processing 10.1016/j.ecoinf.2023.102060...
7/189: Processing 10.1145/3054977.3054986...
8/189: Processing 10.1016/j.cub.2023.05.032...
9/189: Processing 10.7717/peerj.13837...
10/189: Processing 10.1038/s41592-022-01426-1...
11/189: Processing 10.1038/s41598-023-50308-9...
12/189: Processing 10.7554/elife.58145...
13/189: Processing 10.1007/s42991-021-00168-5...
14/189: Processing 10.1016/j.gecco.2023.e02511...
15/189: Processing 10.1038/s41598-024-78509-w...
16/189: Processing 10.1145/3372278.3390710...
17/189: Processing 10.1002/rse2.421...
18/189: Processing 10.1371/journal.pone.0136487...
19/189: Processing 10.1007/s10661-020-08653-z...
20/189: Processing 10.1038/s41597-022-01627-5...
21/189: Processing 10.1002/rse2.17...
22/189: Processing 10.3390/s22249600...
23/189: Processing 10.3390/insects1509072

In [315]:
summary_df

,overall_similarity,Species_vs_Species,Country - verified_vs_Country - verified,Imaging Method (Text) - verified_vs_Imaging Method (new) - verified,Light Spectra_vs_Light Spectra,CV Tasks - verified_vs_CV Tasks - verified,CV Algorithms - verified_vs_CV Algorithms - verified,Dataset - verified_vs_Dataset - verified,Dataset Clustered - verified_vs_Dataset Clustered - verified,ParentHabitat values_vs_ParentHabitat values
doi,,,,,,,,,,
10.1038/s41598-023-48635-y,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
10.1007/s10661-020-08653-z,0.988644,1.000000,1.000000,1.000000,1.000000,1.000000,0.897797,1.000000,1.000000,1.000000
10.1016/j.ecoinf.2024.102679,0.985808,0.984975,1.000000,1.000000,1.000000,1.000000,0.887300,1.000000,1.000000,1.000000
10.1049/ipr2.13090,0.982785,0.968481,1.000000,1.000000,1.000000,1.000000,0.876586,1.000000,1.000000,1.000000
10.1145/3702336.3702340,0.968794,1.000000,1.000000,1.000000,0.719147,1.000000,1.000000,1.000000,1.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...
10.1002/rse2.378,0.635311,0.815967,0.307652,1.000000,1.000000,0.539455,0.248455,0.137321,0.668948,1.000000
10.3390/ani13091526,0.623563,0.781824,1.000000,0.157957,0.254136,1.000000,0.715910,0.033289,0.668948,1.000000
10.1016/j.applanim.2023.106024,0.578704,0.320858,1.000000,1.000000,0.351639,1.000000,0.617259,0.066752,0.668948,0.182883


In [316]:
summary_df.mean().to_frame().T.mean()

overall_similarity                                                     0.797511
Species_vs_Species                                                     0.870214
Country - verified_vs_Country - verified                               0.807813
Imaging Method (Text) - verified_vs_Imaging Method (new) - verified    0.922011
Light Spectra_vs_Light Spectra                                         0.744567
CV Tasks - verified_vs_CV Tasks - verified                             0.888999
CV Algorithms - verified_vs_CV Algorithms - verified                   0.705735
Dataset - verified_vs_Dataset - verified                               0.566510
Dataset Clustered - verified_vs_Dataset Clustered - verified           0.936943
ParentHabitat values_vs_ParentHabitat values                           0.734811
dtype: float64